# 누락·분리·정션 구조 지표

설문·추가 학습 없이 이미 계산한 missing/split/junction 세 값만 0~100 지표로 바꿉니다.
높을수록 세 오류가 적습니다. 사람 선호나 글자 품질에 교정한 점수는 아닙니다.

1. 첫 셀에서 입력과 가중치/변환 크기를 설정합니다.
2. 모듈 로드 셀과 계산 셀을 실행합니다.
3. 글자 structural_score와 획별 stroke_score를 확인합니다.

기존 구조 진단 노트북에 추가 셀을 붙이면 structure_results를 바로 재사용합니다.
이 독립 파일을 새로 열었다면 REPORT_PATH를 결과 JSON 경로로 지정하세요.
MODE="values"는 숫자만 입력하는 테스트입니다. 기존 추출 모델을 재실행하지 않습니다.

획 점수 = 100 × (1-M)^wM × exp(-wD·D/sD-wJ·J/sJ)

글자 점수 = (1-최악획비중) × 획 평균 + 최악획비중 × 최저 획 점수

기본값에서 누락 1인 획은 0점, 세 오류가 0인 획은 100점입니다.
정션 측정의 불확실성은 점수에도 전달됩니다. 노란 후보 수·파임·설문값은 공식에 넣지 않습니다.


In [ ]:
# ===== 점수 설정: 누락·분리·정션 세 값만 사용 =====
# auto: 현재 structure_results가 있으면 재사용, 없으면 REPORT_PATH 읽기
# values: 아래 VALUES의 숫자만으로 바로 계산
MODE = "auto"
REPORT_PATH = "/content/drive/Shareddrives/2026 자율연구/HSQM/stroke_zip_runs/20261028_082616_066020/structure_20261101_092625_060835/structure_report.json"
SAVE_DIR = None  # None = 원래 결과 폴더 아래 indicator_시간. 직접 지정 시 새 폴더

SCORE_CONFIG = dict(
    missing_weight=1.0,   # 누락 중요도. 0이면 이 항 제외
    split_weight=1.0,     # 분리 중요도
    junction_weight=1.0,  # 정션 중요도
    split_scale=0.2,      # 분리값이 이 크기일 때 이 항의 유지율은 exp(-1)
    junction_scale=0.01,  # 정션값이 이 크기일 때 이 항의 유지율은 exp(-1)
    worst_stroke_weight=0.5, # 글자 점수: 획 평균과 최악 획 점수의 혼합. 0=평균, 1=최악
    eps=1e-6,
)

# MODE="values"일 때 사용. 이미지나 모델 없이 이 숫자만 계산합니다.
VALUES = [
    dict(stroke=1, missing=0.0, split=0.0, junction=0.0),
    dict(stroke=2, missing=0.809, split=0.0, junction=0.0),
]


In [ ]:
# 점수 모듈 내장: 추가 모델·가중치 다운로드 없이 실행
import sys, types
_score_module = types.ModuleType("_structural_indicator_runtime")
_score_module.__file__ = "<embedded:structural_indicator>"
sys.modules[_score_module.__name__] = _score_module
exec(compile('"""Deterministic structural indicator using ONLY missing, split and junction.\n\nNo survey, fitted parameters, image embeddings, width or hard branch counts.\nThe 0..100 scale is a configurable engineering index, not calibrated quality.\n"""\nfrom dataclasses import asdict, dataclass\nfrom datetime import datetime\nimport json\nimport math\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\n\n\n@dataclass(frozen=True)\nclass IndicatorConfig:\n    missing_weight: float = 1.0\n    split_weight: float = 1.0\n    junction_weight: float = 1.0\n    split_scale: float = 0.2\n    junction_scale: float = 0.01\n    worst_stroke_weight: float = 0.5\n    eps: float = 1e-6\n\n    def __post_init__(self):\n        for name, value in asdict(self).items():\n            if not math.isfinite(value):\n                raise ValueError(f\'{name} must be finite\')\n        if min(self.missing_weight, self.split_weight, self.junction_weight) < 0:\n            raise ValueError(\'Weights must be nonnegative\')\n        if self.split_scale <= 0 or self.junction_scale <= 0 or not 0 < self.eps < 1:\n            raise ValueError(\'Scales must be positive; eps must be in (0,1)\')\n        for weight, scale in ((self.split_weight, self.split_scale), (self.junction_weight, self.junction_scale)):\n            if not math.isfinite(weight/scale) or weight/scale > torch.finfo(torch.float32).max:\n                raise ValueError(\'weight/scale must fit a finite float32 coefficient\')\n        if not 0 <= self.worst_stroke_weight <= 1:\n            raise ValueError(\'worst_stroke_weight must be in [0,1]\')\n\n\nclass StructuralIndicator(nn.Module):\n    """Map M,D,J to stroke and glyph scores without training.\n\n    Inputs: K or B,K. Missing is in [0,1]; split/junction are nonnegative.\n    q_k = 100*(1-M_k)^wM * exp(-wD*D_k/sD-wJ*J_k/sJ)\n    Q = (1-alpha)*mean(q_k) + alpha*min(q_k), over active reference strokes.\n    min has subgradients; the full pipeline is not everywhere smooth.\n    """\n    def __init__(self, config=None):\n        super().__init__()\n        self.config = config or IndicatorConfig()\n\n    def forward(self, missing, split, junction, active=None):\n        missing = torch.as_tensor(missing)\n        if not missing.is_floating_point():\n            missing = missing.float()\n        split = torch.as_tensor(split, device=missing.device, dtype=missing.dtype)\n        junction = torch.as_tensor(junction, device=missing.device, dtype=missing.dtype)\n        if missing.ndim not in (1, 2) or missing.shape != split.shape or missing.shape != junction.shape:\n            raise ValueError(\'missing, split, junction must have matching K or B,K shapes\')\n        if active is None:\n            active = torch.ones_like(missing, dtype=torch.bool)\n        if active.dtype != torch.bool or active.shape != missing.shape or active.device != missing.device:\n            raise ValueError(\'active must be a bool tensor matching the input shape/device\')\n        if missing.ndim == 1:\n            missing, split, junction, active = [x[None] for x in (missing, split, junction, active)]\n        for name, value in [(\'missing\', missing), (\'split\', split), (\'junction\', junction)]:\n            if not bool(torch.isfinite(value).all()) or bool((value < 0).any()):\n                raise ValueError(f\'{name} must be finite and nonnegative\')\n        if bool((missing > 1).any()) or not bool(active.any(1).all()):\n            raise ValueError(\'Missing must be <=1 and each sample needs an active stroke\')\n        cfg = self.config\n        # Smooth endpoint extension: exact 0/1 retention with finite gradients,\n        # including fractional weights. Unlike a hard zero branch, full missing\n        # still receives a restorative gradient at the default weight of one.\n        if cfg.missing_weight == 0:\n            retain_m = 1 + missing*0\n        else:\n            epsilon = missing.new_tensor(cfg.eps)\n            lower = epsilon.pow(cfg.missing_weight)\n            denominator = (1+epsilon).pow(cfg.missing_weight)-lower\n            retain_m = ((1-missing+epsilon).pow(cfg.missing_weight)-lower)/denominator\n        retain_d = torch.exp(-(cfg.split_weight/cfg.split_scale) * split)\n        retain_j = torch.exp(-(cfg.junction_weight/cfg.junction_scale) * junction)\n        stroke_scores = 100 * retain_m * retain_d * retain_j\n        mean = (stroke_scores * active).sum(1) / active.sum(1)\n        worst = stroke_scores.masked_fill(~active, torch.inf).amin(1)\n        score = (1-cfg.worst_stroke_weight)*mean + cfg.worst_stroke_weight*worst\n        return {\'score\': score, \'stroke_scores\': stroke_scores, \'mean_score\': mean,\n                \'worst_score\': worst, \'penalty\': 1-score/100, \'active\': active,\n                \'retention\': {\'missing\': retain_m, \'split\': retain_d, \'junction\': retain_j}}\n\n\ndef score_structure_report(report, config=None, save_dir=None):\n    """Score an existing structure_report.json/dict without rerunning extraction.\n\n    Returns an auditable report and saves glyph_scores.csv, stroke_scores.csv,\n    indicator_report.json. Only the three named error columns enter the formula.\n    """\n    import pandas as pd\n\n    cfg = config or IndicatorConfig()\n    source_path = None\n    if isinstance(report, (str, Path)):\n        source_path = Path(report)\n        report = json.loads(source_path.read_text(encoding=\'utf-8-sig\'))\n    if not isinstance(report, dict) or not report.get(\'results\'):\n        raise ValueError(\'Provide structure_results or a structure_report.json with nonempty results\')\n    if save_dir is None:\n        base = source_path.parent if source_path else Path(report[\'results\'][0].get(\'npz\', \'.\')).parent\n        save_dir = base / (\'indicator_\' + datetime.now().strftime(\'%Y%m%d_%H%M%S_%f\'))\n    destination = Path(save_dir)\n    destination.mkdir(parents=True, exist_ok=False)\n    scorer = StructuralIndicator(cfg)\n    glyph_rows, stroke_rows, results = [], [], []\n    for number, sample in enumerate(report[\'results\'], 1):\n        strokes = sample.get(\'strokes\', [])\n        if not strokes:\n            raise ValueError(f\'Sample {number} has no per-stroke measurements\')\n        missing, split, junction = [torch.tensor([s[key] for s in strokes], dtype=torch.float64)\n                                     for key in (\'missing\', \'split\', \'junction\')]\n        with torch.no_grad():\n            output = scorer(missing, split, junction)\n        identity = {\'sample\': number, \'csv_row\': sample.get(\'csv_row\'),\n                    \'input\': sample.get(\'input\') or sample.get(\'input_npz\') or \'\'}\n        glyph = {**identity, \'structural_score\': float(output[\'score\'][0]),\n                 \'mean_stroke_score\': float(output[\'mean_score\'][0]),\n                 \'worst_stroke_score\': float(output[\'worst_score\'][0]),\n                 \'stroke_count\': len(strokes)}\n        glyph_rows.append(glyph)\n        scored_strokes = []\n        for k, stroke in enumerate(strokes):\n            row = {**identity, \'stroke\': stroke.get(\'stroke\', k+1),\n                   \'missing\': float(missing[k]), \'split\': float(split[k]), \'junction\': float(junction[k]),\n                   \'stroke_score\': float(output[\'stroke_scores\'][0, k]),\n                   **{name+\'_retention\': float(value[0, k]) for name, value in output[\'retention\'].items()}}\n            stroke_rows.append(row)\n            scored_strokes.append(row)\n        results.append({**glyph, \'strokes\': scored_strokes})\n    pd.DataFrame(glyph_rows).to_csv(destination / \'glyph_scores.csv\', index=False, encoding=\'utf-8-sig\')\n    pd.DataFrame(stroke_rows).to_csv(destination / \'stroke_scores.csv\', index=False, encoding=\'utf-8-sig\')\n    output_report = {\'indicator\': \'missing_split_junction_v1\', \'range\': [0, 100],\n                     \'uses_only\': [\'missing\', \'split\', \'junction\'], \'trained\': False,\n                     \'calibrated_to_human_quality\': False, \'config\': asdict(cfg),\n                     \'measurement_config\': report.get(\'config\'), \'source_report\': str(source_path) if source_path else None,\n                     \'save_dir\': str(destination), \'results\': results}\n    (destination / \'indicator_report.json\').write_text(json.dumps(output_report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    print(\'Structural indicator (0..100; higher is better; uncalibrated):\')\n    display = pd.DataFrame(glyph_rows)\n    display[\'input\'] = display[\'input\'].map(lambda p: str(p).replace(\'\\\\\', \'/\').rsplit(\'/\', 1)[-1])\n    print(display.round(2).to_string(index=False))\n    print(\'\\nPer-stroke scores:\')\n    print(pd.DataFrame(stroke_rows)[[\'sample\', \'stroke\', \'missing\', \'split\', \'junction\', \'stroke_score\']]\n          .to_string(index=False, float_format=lambda x: f\'{x:.4g}\'))\n    print(\'Saved:\', destination)\n    return output_report\n', _score_module.__file__, "exec"), _score_module.__dict__)


In [ ]:
from _structural_indicator_runtime import IndicatorConfig, score_structure_report
from pathlib import Path

if MODE == "values":
    indicator_source = {"results": [{"input": "manual_values", "strokes": VALUES}]}
elif MODE == "auto":
    indicator_source = globals().get("structure_results")
    if indicator_source is None:
        if not REPORT_PATH:
            raise ValueError("구조 진단을 먼저 실행하거나 REPORT_PATH에 structure_report.json 경로를 넣으세요.")
        if str(REPORT_PATH).startswith("/content/drive/"):
            from google.colab import drive
            drive.mount("/content/drive")
        indicator_source = Path(REPORT_PATH)
else:
    raise ValueError('MODE는 "auto" 또는 "values"여야 합니다.')

indicator_results = score_structure_report(
    indicator_source, config=IndicatorConfig(**SCORE_CONFIG), save_dir=SAVE_DIR,
)
